In [2]:
# 1. Import Libraries
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# 2. Load Datasets
ratings = pd.read_csv("data/ratings.csv")

# Load MovieLens-TMDB ID mapping
links = pd.read_csv("data/links.csv")

# Load TMDB movie information
tmdb = pd.read_csv("data/tmdb_5000_movies.csv")

print("Ratings Dataset:")
display(ratings.head())

print("\nLinks Dataset:")
display(links.head())

print("\nTMDB Dataset:")
display(
    tmdb[
        ["id", "title", "vote_average", "vote_count"]
    ].head()
)

print("\nTMDB Columns:")
print(tmdb.columns)

Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931



Links Dataset:


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0



TMDB Dataset:


,id,title,vote_average,vote_count
0,19995,Avatar,7.2,11800
1,285,Pirates of the Caribbean: At World's End,6.9,4500
2,206647,Spectre,6.3,4466
3,49026,The Dark Knight Rises,7.6,9106
4,49529,John Carter,6.1,2124



TMDB Columns:
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')


In [4]:
# 3. Check Columns
print("Ratings Columns:")
print(ratings.columns.tolist())

print("\nLinks Columns:")
print(links.columns.tolist())

print("\nTMDB Columns:")
print(tmdb.columns.tolist())

Ratings Columns:
['userId', 'movieId', 'rating', 'timestamp']

Links Columns:
['movieId', 'imdbId', 'tmdbId']

TMDB Columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [5]:
# 4. Merge Ratings with Links
ratings_links = pd.merge(
    ratings,
    links,
    on="movieId",
    how="inner"
)

print("Ratings + Links:")
display(ratings_links.head())

print("Shape:", ratings_links.shape)

Ratings + Links:


,userId,movieId,rating,timestamp,imdbId,tmdbId
0,1,1,4.0,964982703,114709,862.0
1,1,3,4.0,964981247,113228,15602.0
2,1,6,4.0,964982224,113277,949.0
3,1,47,5.0,964983815,114369,807.0
4,1,50,5.0,964982931,114814,629.0


Shape: (100836, 6)


In [6]:
# 5. Merge with TMDB
merged_data = pd.merge(
    ratings_links,
    tmdb,
    left_on="tmdbId",
    right_on="id",
    how="inner"
)

print("Final Merged Dataset:")

display(
    merged_data[
        [
            "userId",
            "movieId",
            "rating",
            "tmdbId",
            "title",
            "genres",
            "vote_average",
            "vote_count"
        ]
    ].head()
)

print("Shape:", merged_data.shape)


Final Merged Dataset:


,userId,movieId,rating,tmdbId,title,genres,vote_average,vote_count
0,1,1,4.0,862.0,Toy Story,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 35, ""...",7.7,5269
1,1,47,5.0,807.0,Se7en,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 9648, ""na...",8.1,5765
2,1,50,5.0,629.0,The Usual Suspects,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 80, ""name...",8.1,3254
3,1,70,3.0,755.0,From Dusk Till Dawn,"[{""id"": 27, ""name"": ""Horror""}, {""id"": 28, ""nam...",6.9,1603
4,1,101,5.0,13685.0,Bottle Rocket,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 80, ""nam...",6.8,281


Shape: (70194, 26)


In [7]:
# 6. Create Collaborative Filtering Dataset
cf_data = merged_data[
    [
        "userId",
        "movieId",
        "title",
        "rating",
        "vote_average",
        "vote_count"
    ]
].copy()

print("Collaborative Filtering Dataset:")

display(cf_data.head())

print("Shape:", cf_data.shape)


Collaborative Filtering Dataset:


,userId,movieId,title,rating,vote_average,vote_count
0,1,1,Toy Story,4.0,7.7,5269
1,1,47,Se7en,5.0,8.1,5765
2,1,50,The Usual Suspects,5.0,8.1,3254
3,1,70,From Dusk Till Dawn,3.0,6.9,1603
4,1,101,Bottle Rocket,5.0,6.8,281


Shape: (70194, 6)


In [8]:
# 7. Movie Statistics
movie_stats = pd.DataFrame(
    cf_data.groupby("title")["rating"].mean()
)

movie_stats["number of ratings"] = (
    cf_data.groupby("title")["rating"].count()
)

movie_stats["vote_average"] = (
    cf_data.groupby("title")["vote_average"].first()
)

movie_stats["vote_count"] = (
    cf_data.groupby("title")["vote_count"].first()
)

display(movie_stats.head())

,rating,number of ratings,vote_average,vote_count
title,,,,
(500) Days of Summer,3.666667,42,7.2,2904
10 Cloverfield Lane,3.678571,14,6.8,2468
10 Things I Hate About You,3.527778,54,7.3,1701
102 Dalmatians,2.777778,9,5.1,313
10th & Wolf,4.500000,1,6.3,24


In [9]:
# 8. Movies with Most User Ratings
top_rated_movies = (
    movie_stats
    .sort_values(
        "number of ratings",
        ascending=False
    )
    .head(10)
)

display(top_rated_movies)

,rating,number of ratings,vote_average,vote_count
title,,,,
Forrest Gump,4.164134,329,8.2,7927
The Shawshank Redemption,4.429022,317,8.5,8205
Pulp Fiction,4.197068,307,8.3,8428
The Silence of the Lambs,4.161290,279,8.1,4443
The Matrix,4.192446,278,7.9,8907
Star Wars,4.231076,251,8.1,6624
Jurassic Park,3.750000,238,7.6,4856
Braveheart,4.031646,237,7.7,3336
Terminator 2: Judgment Day,3.970982,224,7.7,4185


In [10]:
# 9. Create User-Movie Matrix
movie_matrix = cf_data.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

movie_matrix.head()

title,(500) Days of Summer,10 Cloverfield Lane,10 Things I Hate About You,102 Dalmatians,10th & Wolf,11:14,12 Angry Men,12 Rounds,12 Years a Slave,127 Hours,...,Zoolander,Zoolander 2,Zoom,Zulu,[REC],[REC]Â²,eXistenZ,xXx,xXx: State of the Union,Ã†on Flux
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 10. Enter Favorite Movie
favorite_movie = input(
    "Please enter your favorite movie: "
)

print("Selected Movie:", favorite_movie)

Selected Movie: Toy Story


In [12]:
# 11. Calculate Correlation
similar = movie_matrix.corrwith(
    movie_matrix[favorite_movie]
)

corr = pd.DataFrame(
    similar,
    columns=["Correlation"]
)

corr.dropna(inplace=True)

corr.head()

c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


,Correlation
title,
(500) Days of Summer,0.353833
10 Cloverfield Lane,-0.285732
10 Things I Hate About You,0.322741
102 Dalmatians,-0.637459
11:14,0.000000


In [13]:
# 12. Add Movie Statistics
corr = corr.join(
    movie_stats[
        [
            "number of ratings",
            "vote_average",
            "vote_count"
        ]
    ]
)

corr.head()

,Correlation,number of ratings,vote_average,vote_count
title,,,,
(500) Days of Summer,0.353833,42,7.2,2904
10 Cloverfield Lane,-0.285732,14,6.8,2468
10 Things I Hate About You,0.322741,54,7.3,1701
102 Dalmatians,-0.637459,9,5.1,313
11:14,0.000000,4,6.8,206


In [18]:
# 13. Filter Movies
recommendations = corr[
    corr["number of ratings"] >= 100
].copy()

recommendations = recommendations.sort_values(
    "Correlation",
    ascending=False
)

recommendations = recommendations.drop(
    favorite_movie
)

recommendations.head(10)

,Correlation,number of ratings,vote_average,vote_count
title,,,,
The Incredibles,0.643301,125,7.4,5152
Finding Nemo,0.618701,141,7.6,6122
Aladdin,0.611892,183,7.4,3416
"Monsters, Inc.",0.490231,132,7.5,5996
Mrs. Doubtfire,0.446261,144,7.0,1591
AmÃ©lie,0.438237,120,7.8,3310
American Pie,0.420117,103,6.4,2296
Die Hard: With a Vengeance,0.410939,144,6.9,2066
E.T. the Extra-Terrestrial,0.409216,122,7.3,3269


In [19]:
# 14. Top 10 Recommendations
top_recommendations = (
    recommendations
    .head(10)
    .reset_index()
)

top_recommendations = top_recommendations.rename(
    columns={
        "title": "Movie Title",
        "Correlation": "Correlation Score",
        "number of ratings": "Number of Ratings",
        "vote_average": "TMDB Rating",
        "vote_count": "TMDB Vote Count"
    }
)

top_recommendations

,Movie Title,Correlation Score,Number of Ratings,TMDB Rating,TMDB Vote Count
0,The Incredibles,0.643301,125,7.4,5152
1,Finding Nemo,0.618701,141,7.6,6122
2,Aladdin,0.611892,183,7.4,3416
3,"Monsters, Inc.",0.490231,132,7.5,5996
4,Mrs. Doubtfire,0.446261,144,7.0,1591
5,AmÃ©lie,0.438237,120,7.8,3310
6,American Pie,0.420117,103,6.4,2296
7,Die Hard: With a Vengeance,0.410939,144,6.9,2066
8,E.T. the Extra-Terrestrial,0.409216,122,7.3,3269
9,Home Alone,0.408444,116,7.1,2414
